In [1]:
import numpy as np
import xarray as xr
import dask.array as da


In [2]:

# Create the data array
data = xr.DataArray(
    np.random.rand(5, 6),
    dims=["time", "variable"],
    coords={
        "time": np.arange(5),
        "variable": np.arange(6),
    },
)

In [4]:
%%timeit
# Define slices
slices = [slice(0, 2), slice(1, 3), slice(2, 4)] # Same length and continuous slices

# Select the data for each slice
sliced_data = [data.isel(time=slc).assign_coords(time=np.arange(slc.stop - slc.start)) for slc in slices]

# Concatenate along the new dimension 'window_dim'
result = xr.concat(sliced_data, dim='window_dim')
# print(result)

3.54 ms ± 1.02 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
%%timeit
# Define slices
window_size = 2
step = 1
slices = np.arange(0, data.sizes['time'] - window_size + 1, step)

# Use advanced indexing to select overlapping slices
sliced_data = data.isel(time=xr.DataArray(slices[:, None] + np.arange(window_size), dims=['window_dim', 'window_step']))

# Adjust the coordinates
result = sliced_data.assign_coords(time=sliced_data.time - slices[:, None])
# print(result)

585 µs ± 91.7 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [6]:
%%timeit
# Define slices
window_size = 2
step = 1
slices = np.arange(0, data.sizes['time'] - window_size + 1, step)

# Use advanced indexing to select overlapping slices
sliced_data = data.isel(time=xr.DataArray(slices[:, None] + np.arange(window_size), dims=['window_dim', 'window_step']))

# Adjust the coordinates
result = sliced_data.assign_coords(time=sliced_data.time - slices[:, None])

# Trigger computation
result.compute()
# print(result)

691 µs ± 123 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
